# Weak perturbation — MHD (fluid) case

B1: small-amplitude velocity perturbation propagating as sound wave. Compare measured phase speed to theory $c_s = \sqrt{\gamma T} = \sqrt{5/3} \approx 1.29$.

In [ ]:
import os
import sys
from pathlib import Path

try: # notebooks has no __file__
    WORK_DIR  = Path(__file__).parent 
except NameError:
    WORK_DIR  = Path(os.getcwd()) 
PROJECT_DIR = Path(WORK_DIR).parent.parent.parent.parent.parent
work_path = str(WORK_DIR)

In [ ]:
paths = [str(PROJECT_DIR/s) for s in ["", "build", "pyphare"]]
sys.path.extend(paths)
os.environ['PYTHONPATH'] = os.pathsep.join(paths) + os.pathsep + os.environ.get('PYTHONPATH', "")

In [ ]:
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from pyphare.pharesee.run import Run

In [ ]:
run_name = "B1"
run_path = os.path.join(str(work_path), run_name)
os.makedirs(run_path, exist_ok=True)

In [ ]:
if "mhd_rho.h5" not in os.listdir(run_path):
    result = subprocess.run(
        ["mpirun", "-n", "2", "python3", "-Ou",
         str(work_path) + "/weak_mhd.py", run_path],
        env=os.environ, check=True
    )

In [ ]:
run = Run(run_path)

In [ ]:
times = [1.6, 3.2, 4.8, 6.4, 8.0]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

fig, ax = plt.subplots(figsize=(10, 4))
for i, t in enumerate(times):
    V = run.GetMHDV(t)
    V.plot(qty='x', ax=ax, ls='solid', lw=1.5, color=colors[i], label=f"t={t}")
ax.set_xlabel("x")
ax.set_ylabel("Vx")
ax.legend()
plt.show()

In [ ]:
import pyphare.pharesee as phc
from pyphare.pharesee.hierarchy import uniformgrid as uniform
from numpy import polyfit

fit_times = [1.6, 3.2, 4.8, 6.4]
pks_plus = []
pks_minus = []

fig, ax = plt.subplots(figsize=(12, 3))
for i, t in enumerate(times):
    V = run.GetMHDV(t)
    v = phc.filters.gaussian(V, sigma=6)

    if t in fit_times:
        coords = uniform.peak_coordinates(v, 'x', height=0.01, distance=20)
        if len(coords) == 1:
            pks_minus.append(coords[0])
            pks_plus.append(coords[0])
        else:
            pks_minus.append(coords[1])
            pks_plus.append(coords[0])

    v.plot(qty='x', ax=ax, ls='solid', lw=2.0, color=colors[i], label=f"t={t}")
    for p in uniform.peak_coordinates(v, 'x', height=0.01, distance=20):
        ax.axvline(x=p, color='black', linestyle='dotted')

ax.set_xlabel("x")
ax.set_ylabel("Vx")
ax.legend()
plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 3))
slope_m, origin_m = polyfit(fit_times, pks_minus, 1)
slope_p, origin_p = polyfit(fit_times, pks_plus, 1)

ax2.plot(fit_times, pks_minus, marker="o", color=colors[5], ls="None")
ax2.plot(fit_times, np.array(fit_times)*slope_m + origin_m, color=colors[5])
ax2.plot(fit_times, pks_plus, marker="o", color=colors[6], ls="None")
ax2.plot(fit_times, np.array(fit_times)*slope_p + origin_p, color=colors[6])
ax2.text(1.6, 6.4, 'positive slope : {:.3f}'.format(slope_p))
ax2.text(1.6, 1.2, 'negative slope : {:.3f}'.format(slope_m))
ax2.set_ylim([0, 8])
ax2.set_xlabel('Time')
ax2.set_ylabel('X-position')
plt.show()

v_phi = np.mean(np.abs([slope_m, slope_p]))
print(f"measured phase speed: {v_phi:.3f}  theory: {np.sqrt(5/3*1.0):.3f}")


## Theory

The MHD sound speed for an ideal gas is
$$
c_s = \sqrt{\frac{\gamma p_0}{\rho_0}}
$$
With $\gamma = 5/3$, $p_0 = 1$, $\rho_0 = 1$ this gives $c_s = \sqrt{5/3} \approx 1.291$.

The initial Gaussian velocity bump splits into two counter-propagating pulses of equal amplitude, one moving in the $+x$ direction and one in the $-x$ direction. Each travels at the sound speed $c_s$. Because the domain is periodic, both pulses re-enter from the opposite boundary after crossing the domain.

## Sample questions

1. What is the expected phase speed of the sound wave?
2. How does the wave amplitude evolve over time?
3. What happens if you increase the initial amplitude beyond $c_s$?